In [2]:
import pandas as pd
from pandasql import sqldf

In [3]:
users = pd.DataFrame({
    'user_id': [1,2,3,4],
    'name': ['Аня','Боря','Вера','Гена']
})
orders = pd.DataFrame({
    'order_id': [101, 102, 103],
    'user_id': [1, 1, 2],
    'amount': [500, 300, 700]
})

sqldf("SELECT users.name, orders.amount FROM users INNER JOIN orders ON users.user_id = orders.user_id")

,name,amount
0,Аня,300
1,Аня,500
2,Боря,700


In [4]:
sqldf("SELECT users.name, orders.amount FROM users LEFT JOIN orders ON users.user_id = orders.user_id")

,name,amount
0,Аня,300.0
1,Аня,500.0
2,Боря,700.0
3,Вера,NaN
4,Гена,NaN


In [5]:
sqldf("SELECT users.name FROM users LEFT JOIN orders ON users.user_id = orders.user_id WHERE orders.order_id is NULL")

,name
0,Вера
1,Гена


In [6]:
reviews = pd.read_csv('/home/art/olist_data/olist_order_reviews_dataset.csv')
reviews.head()
order_items = pd.read_csv('/home/art/olist_data/olist_order_items_dataset.csv')

In [7]:
reviews.shape

(99224, 7)

In [8]:
orders = pd.read_csv('/home/art/olist_data/olist_orders_dataset.csv')

sqldf("""
    SELECT orders.order_id, orders.order_status, reviews.review_score
    FROM orders
    LEFT JOIN reviews ON orders.order_id = reviews.order_id
    LIMIT 10
""")

,order_id,order_status,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,4
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,4
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,5
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,5
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,5
5,a4591c265e18cb1dcee52889e2d8acc3,delivered,4
6,136cce7faa42fdb2cefd53fdc79a6098,invoiced,2
7,6514b8ad8028c9f2cc2374ded245783f,delivered,5
8,76c6e866289321a7c93b82b54852dc33,delivered,1
9,e69bfb5eb88e0ed6a785585b27e16dbf,delivered,5


In [9]:
sqldf("""
    SELECT COUNT(*) AS orders_without_reviews
    FROM orders
    LEFT JOIN reviews ON orders.order_id = reviews.order_id
    WHERE reviews.review_id IS NULL
""")

,orders_without_reviews
0,768


In [10]:
sqldf("""
    SELECT COUNT(*) AS orders_with_reviews
    FROM orders
    INNER JOIN reviews ON orders.order_id = reviews.order_id
""")

,orders_with_reviews
0,99224


In [11]:
sqldf("SELECT AVG(review_score) AS avg_score FROM reviews")

,avg_score
0,4.086421


In [12]:

customers = pd.read_csv('/home/art/olist_data/olist_customers_dataset.csv')

sqldf("""
    SELECT 
        customers.customer_state,
        SUM(order_items.price) AS revenue
    FROM orders
    JOIN customers ON orders.customer_id = customers.customer_id
    JOIN order_items ON orders.order_id = order_items.order_id
    GROUP BY customers.customer_state
    ORDER BY revenue DESC
    LIMIT 10
""")

,customer_state,revenue
0,SP,5202955.05
1,RJ,1824092.67
2,MG,1585308.03
3,RS,750304.02
4,PR,683083.76
5,SC,520553.34
6,BA,511349.99
7,DF,302603.94
8,GO,294591.95
9,ES,275037.31


In [13]:
sqldf("""
    SELECT
        customers.customer_city,
        COUNT(DISTINCT orders.order_id) AS orders_count
    FROM orders
    JOIN customers ON orders.customer_id = customers.customer_id
    GROUP BY customers.customer_city
    ORDER BY orders_count DESC
    LIMIT 10
""")

,customer_city,orders_count
0,sao paulo,15540
1,rio de janeiro,6882
2,belo horizonte,2773
3,brasilia,2131
4,curitiba,1521
5,campinas,1444
6,porto alegre,1379
7,salvador,1245
8,guarulhos,1189
9,sao bernardo do campo,938


In [14]:
sqldf("""
    SELECT
        customers.customer_state,
        AVG(order_items.price) AS avg_check
    FROM orders
    JOIN customers ON orders.customer_id = customers.customer_id
    JOIN order_items ON orders.order_id = order_items.order_id
    GROUP BY customers.customer_state
    ORDER BY avg_check DESC
    LIMIT 10
""")

,customer_state,avg_check
0,PB,191.475216
1,AL,180.889212
2,AC,173.727717
3,RO,165.973525
4,PA,165.692417
5,AP,164.320732
6,PI,160.358081
7,TO,157.529333
8,RN,156.965936
9,CE,153.758261


In [15]:
sqldf("""
    SELECT
        customers.customer_state,
        COUNT(DISTINCT customers.customer_unique_id) AS unique_customers
    FROM orders
    JOIN customers ON orders.customer_id = customers.customer_id
    GROUP BY customers.customer_state
    ORDER BY unique_customers DESC
    LIMIT 10
""")

,customer_state,unique_customers
0,SP,40302
1,RJ,12384
2,MG,11259
3,RS,5277
4,PR,4882
5,SC,3534
6,BA,3277
7,DF,2075
8,ES,1964
9,GO,1952


In [16]:
reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [17]:
sqldf("""
    SELECT
        COUNT(*) AS five_stars
    FROM reviews
    WHERE review_score = '5'
""")

,five_stars
0,57328


In [18]:
sqldf("""
    SELECT
        COUNT(DISTINCT customers.customer_city) AS unique_city
    FROM customers
""")

,unique_city
0,4119


In [19]:
sqldf("""
    SELECT AVG(order_items.price) AS avg_check
    FROM orders
    JOIN customers ON orders.customer_id = customers.customer_id
    JOIN order_items ON orders.order_id = order_items.order_id
    WHERE customers.customer_state = 'SP'
""")

,avg_check
0,109.653629


In [20]:
sqldf("""
    SELECT
        order_items.seller_id,
        COUNT(*) AS count
    FROM order_items
    GROUP BY seller_id
    ORDER BY count DESC
    LIMIT 5
""")

,seller_id,count
0,6560211a19b47992c3666cc44a7e94c0,2033
1,4a3ca9315b744ce9f8e9374361493884,1987
2,1f50f920176fa81dab994f9023523100,1931
3,cc419e0650a3c5ba77189a1882b7556a,1775
4,da8622b14eb17ae2831f4ac5b9dab84a,1551


In [21]:
sqldf("""
    SELECT
        COUNT(DISTINCT orders.order_id) AS count
    FROM orders
    WHERE order_status = 'canceled' AND order_purchase_timestamp >= '2018-01-01' AND order_purchase_timestamp < '2019-01-01'
""")

,count
0,334


In [22]:
sqldf("""
    SELECT order_id, price
    FROM order_items
    WHERE price > (SELECT AVG(price) AS avg_price FROM order_items)
    LIMIT 10
""")

,order_id,price
0,00018f77f2f0320c557190d7a144bdd3,239.90
1,000229ec398224ef6ca0657da4fc703e,199.00
2,00042b26cf59d7ce69dfabb4e55b4fd9,199.90
3,000576fe39319847cbb9d288c5617fa6,810.00
4,0005a1a1728c9d785b8e2b08b904576c,145.95
5,0009c9a17f916a706d71784483a5d643,639.00
6,000aed2e25dbad2f9ddb70584c5a2ded,144.00
7,00119ff934e539cf26f92b9ef0cdfed8,219.90
8,0011d82c4b53e22e84023405fb467e57,289.00
9,00137e170939bba5a3134e2386413108,397.00


In [23]:
sqldf("""
    SELECT AVG(order_total) AS avg_order_total
    FROM(
        SELECT order_id, SUM(price) AS order_total
        FROM order_items
        GROUP BY order_id
    )
""")

,avg_order_total
0,137.754076


In [24]:
sqldf("""
    SELECT
        order_id,
        price,
        price - (SELECT AVG(price) FROM order_items) AS diff_from_avg
    FROM order_items
    LIMIT 10
""")

,order_id,price,diff_from_avg
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,-61.753739
1,00018f77f2f0320c557190d7a144bdd3,239.90,119.246261
2,000229ec398224ef6ca0657da4fc703e,199.00,78.346261
3,00024acbcdf0a6daa1e931b038114c75,12.99,-107.663739
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,79.246261
5,00048cc3ae777c65dbb7d2a0634bc1ea,21.90,-98.753739
6,00054e8431b9d7675808bcb819fb4a32,19.90,-100.753739
7,000576fe39319847cbb9d288c5617fa6,810.00,689.346261
8,0005a1a1728c9d785b8e2b08b904576c,145.95,25.296261
9,0005f50442cb953dcd1d21e1fb923495,53.99,-66.663739


In [25]:
sqldf("""
    WITH order_totals AS (
        SELECT order_id, SUM(price) AS order_total
        FROM order_items
        GROUP BY order_id
    )
    SELECT AVG(order_total) AS avg_order_total
    FROM order_totals
""")

,avg_order_total
0,137.754076


In [26]:
sqldf("""
    WITH customer_revenue AS (
        SELECT
            orders.customer_id,
            SUM(order_items.price) AS revenue
        FROM orders
        JOIN order_items ON orders.order_id = order_items.order_id
        GROUP BY orders.customer_id
    )
    SELECT customer_id, revenue
    FROM customer_revenue
    ORDER BY revenue DESC
    LIMIT 10
""")

,customer_id,revenue
0,1617b1357756262bfa56ab541c47bc16,13440.0
1,ec5b2ba62e574342386871631fafd3fc,7160.0
2,c6e2731c5b391845f6800c97401a43a9,6735.0
3,f48d464a0baaea338cb25f816991ab1f,6729.0
4,3fd6777bbce08a352fddd04e4a7cc8f6,6499.0
5,05455dfa7cd02f13d132aa7a6a9729c6,5934.6
6,df55c14d1476a9a3467f131269c2477f,4799.0
7,24bbf5fd2f2e1b359ee7de94defc4a15,4690.0
8,e0a2412720e9ea4f26c1ac985f6a7358,4599.9
9,3d979689f636322c62418b6346b1c6d2,4590.0


In [27]:
sqldf("""
    WITH customer_revenue AS (
        SELECT
            orders.customer_id,
            SUM(order_items.price) AS revenue
        FROM orders
        JOIN order_items ON orders.order_id = order_items.order_id
        GROUP BY orders.customer_id
    ),
    customer_state AS (
            SELECT
                customers.customer_id,
                customers.customer_state
            FROM customers
    )
    SELECT
        customer_state.customer_state,
        AVG(customer_revenue.revenue) AS avg_revenue
    FROM customer_revenue
    JOIN customer_state ON customer_revenue.customer_id = customer_state.customer_id
    GROUP BY customer_state.customer_state
    ORDER BY avg_revenue DESC
""")

,customer_state,avg_revenue
0,PB,216.669323
1,AP,198.151471
2,AC,197.320370
3,AL,195.413163
4,RO,186.804211
5,PA,184.482278
6,TO,177.855699
7,PI,176.296308
8,MT,173.259723
9,RN,172.271743


In [28]:
sqldf("""
    WITH customer_revenue AS(
        SELECT
            orders.customer_id,
            SUM(order_items.price) AS revenue
        FROM orders
        JOIN order_items ON orders.order_id = order_items.order_id
        GROUP BY orders.customer_id
    )
    SELECT
        CASE
            WHEN revenue > 1000 THEN 'VIP'
            WHEN revenue BETWEEN 500 AND 1000 THEN 'Middle'
            ELSE 'Regular'
        END AS segmental,
        COUNT(*) AS customers_count
        FROM customer_revenue
        GROUP BY segmental
        ORDER BY customers_count
""")

,segmental,customers_count
0,VIP,943
1,Middle,2698
2,Regular,95025


In [29]:
sqldf("""
    WITH numbered_orders AS (
        SELECT
            customer_unique_id,
            order_id,
            order_purchase_timestamp,
            ROW_NUMBER() OVER (PARTITION BY customer_unique_id ORDER BY order_purchase_timestamp) AS order_num
        FROM orders
        JOIN customers ON orders.customer_id = customers.customer_id
        )
        SELECT customer_unique_id, order_id, order_purchase_timestamp
        FROM numbered_orders
        WHERE order_num = 1
        ORDER BY order_purchase_timestamp DESC
        LIMIT 20
""")

,customer_unique_id,order_id,order_purchase_timestamp
0,87ab9fec999db8bd5774917de3cdf01c,10a045cdf6a5650c21e9cfeb60384c16,2018-10-17 17:30:18
1,9bb92bebd4cb7511e1a02d5e50bc4655,392ed9afd714e3c74767d0c4d3e3f477,2018-09-29 09:13:03
2,c1ee153508c6b785b491443a95ff364e,ea844c92cf978ea23321fa7fe5871761,2018-09-13 09:56:12
3,09687a7b7431a93b5c53b50ba779bf94,87ae60ef8b08ae0e5f903cacb53a6904,2018-09-11 16:45:54
4,0a5dadb73145d29f42f703c97c377ae8,afb61112fb99b07fe17e591c68c0c84c,2018-09-06 10:48:12
5,968fac81e2c44fb6c1e3ac2a45e6a102,4637ca194b6387e2d538dc89b124b0ee,2018-09-03 14:14:25
6,e90598185d2427a35e32ef241a5c04aa,3a3cddda5a7c27851bd96c3313412840,2018-08-31 16:13:44
7,a8dd81392e5eee5d979c629a76abec2a,8d4c637f1accf7a88a4555f02741e606,2018-08-29 16:27:49
8,24ac2b4327e25baf39f2119e4228976a,35a972d7f8436f405b56e36add1a7140,2018-08-29 15:00:37
9,b701bebbdf478f5500348f03aff62121,03ef5dedbe7492bdae72eec50764c43f,2018-08-29 14:52:00


In [30]:
sqldf("""
    WITH customer_revenue AS (
        SELECT orders.customer_id, SUM(order_items.price) AS revenue
        FROM orders
        JOIN order_items ON orders.order_id = order_items.order_id
        GROUP BY orders.customer_id
    )
    SELECT 
        customer_id,
        revenue,
        RANK() OVER (ORDER BY revenue DESC) AS rank
    FROM customer_revenue
    LIMIT 10
""")

,customer_id,revenue,rank
0,1617b1357756262bfa56ab541c47bc16,13440.0,1
1,ec5b2ba62e574342386871631fafd3fc,7160.0,2
2,c6e2731c5b391845f6800c97401a43a9,6735.0,3
3,f48d464a0baaea338cb25f816991ab1f,6729.0,4
4,3fd6777bbce08a352fddd04e4a7cc8f6,6499.0,5
5,05455dfa7cd02f13d132aa7a6a9729c6,5934.6,6
6,df55c14d1476a9a3467f131269c2477f,4799.0,7
7,24bbf5fd2f2e1b359ee7de94defc4a15,4690.0,8
8,e0a2412720e9ea4f26c1ac985f6a7358,4599.9,9
9,3d979689f636322c62418b6346b1c6d2,4590.0,10


In [31]:
sqldf("""
    WITH customer_revenue AS (
        SELECT orders.customer_id, SUM(order_items.price) AS revenue
        FROM orders
        JOIN order_items ON orders.order_id = order_items.order_id
        GROUP BY orders.customer_id
    )
    SELECT 
        customer_id,
        revenue,
        ROW_NUMBER() OVER (ORDER BY revenue DESC) AS rn,
        RANK() OVER (ORDER BY revenue DESC) AS rnk,
        DENSE_RANK() OVER (ORDER BY revenue DESC) AS drnk
    FROM customer_revenue
    LIMIT 10
""")

,customer_id,revenue,rn,rnk,drnk
0,1617b1357756262bfa56ab541c47bc16,13440.0,1,1,1
1,ec5b2ba62e574342386871631fafd3fc,7160.0,2,2,2
2,c6e2731c5b391845f6800c97401a43a9,6735.0,3,3,3
3,f48d464a0baaea338cb25f816991ab1f,6729.0,4,4,4
4,3fd6777bbce08a352fddd04e4a7cc8f6,6499.0,5,5,5
5,05455dfa7cd02f13d132aa7a6a9729c6,5934.6,6,6,6
6,df55c14d1476a9a3467f131269c2477f,4799.0,7,7,7
7,24bbf5fd2f2e1b359ee7de94defc4a15,4690.0,8,8,8
8,e0a2412720e9ea4f26c1ac985f6a7358,4599.9,9,9,9
9,3d979689f636322c62418b6346b1c6d2,4590.0,10,10,10


In [32]:
sqldf("""
    WITH customer_revenue AS (
        SELECT orders.customer_id, SUM(order_items.price) AS revenue
        FROM orders
        JOIN order_items ON orders.order_id = order_items.order_id
        GROUP BY orders.customer_id
    )
    SELECT revenue, COUNT(*) AS cnt
    FROM customer_revenue
    GROUP BY revenue
    HAVING COUNT(*) > 1
    ORDER BY revenue DESC
    LIMIT 10
""")

,revenue,cnt
0,2999.99,2
1,2999.89,5
2,2899.00,2
3,2700.00,2
4,2699.00,2
5,2690.00,2
6,2649.99,2
7,2649.00,3
8,2589.00,2
9,2470.50,2


In [33]:
sqldf("""
    WITH customer_revenue AS (
        SELECT orders.customer_id, SUM(order_items.price) AS revenue
        FROM orders
        JOIN order_items ON orders.order_id = order_items.order_id
        GROUP BY orders.customer_id
    )
    SELECT 
        customer_id,
        revenue,
        ROW_NUMBER() OVER (ORDER BY revenue DESC) AS rn,
        RANK() OVER (ORDER BY revenue DESC) AS rnk,
        DENSE_RANK() OVER (ORDER BY revenue DESC) AS drnk
    FROM customer_revenue
    WHERE revenue = 2999.89
""")


,customer_id,revenue,rn,rnk,drnk
0,3a4e4932f3e6976dd966f1d3c044d49e,2999.89,1,1,1
1,4312e816d88fa8bdb6246878f61cdad5,2999.89,2,1,1
2,8dd4c93bfbebe2a77657d46ef959a7ac,2999.89,3,1,1
3,d7c94e22bfd332eb29b0f5badc3ce103,2999.89,4,1,1
4,fd78e5e3abdc375368456fe738694c00,2999.89,5,1,1


In [34]:
sqldf("""
    WITH monthly_revenue AS (
        SELECT 
            strftime('%Y-%m', orders.order_purchase_timestamp) AS month,
            SUM(order_items.price) AS revenue
        FROM orders
        JOIN order_items ON orders.order_id = order_items.order_id
        GROUP BY month
    )
    SELECT 
        month,
        revenue,
        LAG(revenue) OVER (ORDER BY month) AS prev_revenue
    FROM monthly_revenue
    ORDER BY month
""")

,month,revenue,prev_revenue
0,2016-09,267.36,NaN
1,2016-10,49507.66,267.36
2,2016-12,10.90,49507.66
3,2017-01,120312.87,10.90
4,2017-02,247303.02,120312.87
5,2017-03,374344.30,247303.02
6,2017-04,359927.23,374344.30
7,2017-05,506071.14,359927.23
8,2017-06,433038.60,506071.14
9,2017-07,498031.48,433038.60


In [35]:
sqldf("""
    WITH monthly_revenue AS (
        SELECT 
            strftime('%Y-%m', orders.order_purchase_timestamp) AS month,
            SUM(order_items.price) AS revenue
        FROM orders
        JOIN order_items ON orders.order_id = order_items.order_id
        GROUP BY month
    )
    SELECT 
        month,
        revenue,
        LAG(revenue) OVER (ORDER BY month) AS prev_revenue,
        revenue - LAG(revenue) OVER (ORDER BY month) AS diff,
        ROUND((revenue - LAG(revenue) OVER (ORDER BY month)) * 100.0 / LAG(revenue) OVER (ORDER BY month), 2) AS growth_pct
    FROM monthly_revenue
    ORDER BY month
""")

,month,revenue,prev_revenue,diff,growth_pct
0,2016-09,267.36,NaN,NaN,NaN
1,2016-10,49507.66,267.36,49240.30,18417.23
2,2016-12,10.90,49507.66,-49496.76,-99.98
3,2017-01,120312.87,10.90,120301.97,1103687.80
4,2017-02,247303.02,120312.87,126990.15,105.55
5,2017-03,374344.30,247303.02,127041.28,51.37
6,2017-04,359927.23,374344.30,-14417.07,-3.85
7,2017-05,506071.14,359927.23,146143.91,40.60
8,2017-06,433038.60,506071.14,-73032.54,-14.43
9,2017-07,498031.48,433038.60,64992.88,15.01


In [36]:
sqldf("""
    WITH monthly_revenue AS (
        SELECT 
            strftime('%Y-%m', orders.order_purchase_timestamp) AS month,
            SUM(order_items.price) AS revenue
        FROM orders
        JOIN order_items ON orders.order_id = order_items.order_id
        GROUP BY month
    )
    SELECT 
        month,
        revenue,
        LEAD(revenue) OVER (ORDER BY month) AS next_revenue
    FROM monthly_revenue
    ORDER BY month
""")


,month,revenue,next_revenue
0,2016-09,267.36,49507.66
1,2016-10,49507.66,10.90
2,2016-12,10.90,120312.87
3,2017-01,120312.87,247303.02
4,2017-02,247303.02,374344.30
5,2017-03,374344.30,359927.23
6,2017-04,359927.23,506071.14
7,2017-05,506071.14,433038.60
8,2017-06,433038.60,498031.48
9,2017-07,498031.48,573971.68


In [37]:
print('all ok')

all ok
